# Grounding an agent on verifiable live data

An autonomous agent should not act on world-state it cannot check. A tool call that *succeeds* is not the same as data that is *trustworthy*: the value can be stale, single-sourced, or a provider simply restating its own claim, and a cryptographic signature on it proves only that the bytes were not altered, never that they are true.

**The rule this recipe builds on: signed is not verified.**

This notebook shows a small, reusable pattern: before an OpenAI agent acts on a live datapoint, it (1) **verifies the Ed25519 signature** so it knows the bytes are intact and who produced them, and (2) reads a portable **reliability object** so it knows *how much to believe the value*, then gates its action on that. We use [Dynamic Feed](https://dynamicfeed.ai) as a keyless, signed live-data source because it emits this object on every datapoint, but the pattern is vendor-neutral: any source that returns the same shape works.

The reliability object carries three separable axes:

- **integrity**: a signature (bytes intact, who produced it)
- **reliability**: a graded `confidence`, source count, `verified` flag (how good the data is)
- **vantage**: `independent` vs `producer-reported` (did the producer observe this, or relay its own claim; corroboration is not independence)

Schema and zero-dependency reference validators are open and MIT-licensed: https://github.com/dynamicfeed/df-verify/tree/main/reliability

## Setup

No account is needed for the data source. You only need an OpenAI key for the agent step.

In [ ]:
%pip install --quiet openai cryptography requests

import os, json, base64, requests
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey

# os.environ['OPENAI_API_KEY'] = 'sk-...'   # set your key
MODEL = os.environ.get('OPENAI_MODEL', 'gpt-4.1')   # set to whichever current model you use
DF = 'https://dynamicfeed.ai'

## 1. The verifier: fetch, check the signature, read reliability

This is the whole trust step, with no dependency on the provider at runtime beyond fetching its public key. It re-canonicalizes the response exactly as it was signed (`json-sorted-compact`), checks the detached Ed25519 signature against the published key, and returns the value alongside its reliability object. Change one byte and verification fails.

In [ ]:
def _b64u(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + '=' * (-len(s) % 4))

def fetch_verified(tool: str, **args) -> dict:
    """Fetch a live datapoint, verify its Ed25519 signature, and return {value, reliability, integrity}."""
    doc = requests.get(f'{DF}/v1/facts', params={'tool': tool, **args}, timeout=20).json()
    sig = doc.get('signature') or {}

    # 1) integrity: re-canonicalize the payload (signature field stripped) and verify the detached signature
    payload = {k: v for k, v in doc.items() if k != 'signature'}
    canon = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True).encode()
    keys = requests.get(f'{DF}/.well-known/keys', timeout=20).json()
    pub = keys.get(sig.get('key_id'))
    integrity_ok = False
    try:
        Ed25519PublicKey.from_public_bytes(_b64u(pub)).verify(_b64u(sig['sig']), canon)
        integrity_ok = True
    except Exception:
        integrity_ok = False

    fact = (doc.get('facts') or [{}])[0]
    return {
        'value': fact.get('value'),
        'reliability': fact.get('reliability') or {},
        'integrity_ok': integrity_ok,   # signature valid: bytes intact
        'entity': fact.get('entity'),
    }

### Try it (no OpenAI key needed)

The `gate` below is the point: integrity alone never promotes a reading to *act*. A single-source, producer-reported value is honest about being unverified.

In [ ]:
def gate(r: dict, require_independent: bool = False) -> str:
    """Decide whether an agent should act on a verified fetch result."""
    if not r['integrity_ok']:
        return 'REFUSE: signature did not verify (data may be altered)'
    rel = r['reliability']
    if require_independent and rel.get('vantage') != 'independent':
        return 'ABSTAIN: producer-reported; an independent vantage was required'
    if rel.get('verified') is True:
        return 'ACT: verified (corroborated by 2+ sources)'
    return f"CAUTION: signed + {rel.get('confidence')} but verified={rel.get('verified')} (vantage={rel.get('vantage')}, sources={rel.get('sources')}); seek a second source before acting"

r = fetch_verified('current_weather', city='Tokyo')
print('value      :', r['value'].get('temperature_c'), 'C' if isinstance(r['value'], dict) else r['value'])
print('integrity  :', r['integrity_ok'])
print('reliability:', {k: r['reliability'].get(k) for k in ('confidence','verified','vantage','sources','basis')})
print('decision   :', gate(r))

## 2. Give it to an OpenAI agent as a tool

Now the agent calls the verifier as a function. The function returns the value **and** its reliability, and the system prompt tells the model the rule: state confidence, and do not act on an unverified reading without flagging it. The model reasons over `verified` and `vantage`, not just the number.

In [ ]:
from openai import OpenAI
client = OpenAI()

tools = [{
    'type': 'function',
    'name': 'get_verified_fact',
    'description': 'Fetch a live datapoint with its cryptographic integrity check and reliability grade.',
    'parameters': {
        'type': 'object',
        'properties': {
            'tool': {'type': 'string', 'description': 'e.g. current_weather, bitcoin_network, earthquakes'},
            'city': {'type': 'string', 'description': 'optional, for location tools'},
        },
        'required': ['tool'],
        'additionalProperties': True,
    },
}]

SYSTEM = (
    'You are an agent that acts on live data. Before relying on a value, call get_verified_fact. '
    'It returns integrity_ok (signature valid) and a reliability object (confidence, verified, vantage). '
    'Rule: signed is not verified. State the confidence and vantage. If verified is not true, say so and '
    'treat the value as provisional rather than fact. Refuse to rely on it if integrity_ok is false.'
)

In [ ]:
def run(user_msg: str) -> str:
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user_msg}]
    resp = client.responses.create(model=MODEL, input=msgs, tools=tools)
    # resolve any function calls, then ask for the final answer
    calls = [o for o in resp.output if getattr(o, 'type', None) == 'function_call']
    if not calls:
        return resp.output_text
    msgs += resp.output
    for c in calls:
        args = json.loads(c.arguments)
        result = fetch_verified(**args)
        msgs.append({'type': 'function_call_output', 'call_id': c.call_id,
                     'output': json.dumps({k: result[k] for k in ('value','reliability','integrity_ok')})})
    final = client.responses.create(model=MODEL, input=msgs, tools=tools)
    return final.output_text

print(run('What is the temperature in Tokyo right now, and should I trust that reading?'))

## Takeaway

The pattern is three lines of discipline an agent can apply to any tool output:

1. **Verify integrity** (the signature) so you know the bytes are intact.
2. **Read reliability** (`confidence`, `verified`, `vantage`) so you know how much to believe the value.
3. **Gate the action** on it, never letting a signature alone stand in for truth.

The reliability object is small, portable, and open (MIT): schema and reference validators at https://github.com/dynamicfeed/df-verify/tree/main/reliability . The same shape is being adopted across agent and data standards, so an agent that learns to read it once can apply it to many sources.